# Switched-Array Radio Direction-Finding System

This notebook implements the complete hardware radio direction-finding system.

The system uses a permanent reference antenna connected to Pluto RX1 and a
set of antennas that are sequentially switched into Pluto RX2. An STM32
controls the RF switch, while Julia coordinates the switch states, acquires
the two receive channels, processes the I/Q data and estimates the direction
of the received signal.

The implementation is kept configurable so that the number of switched
antennas, operating frequency and array geometry can be changed without
restructuring the processing code.

In [ ]:
# Python environment used by pyadi-iio

pluto_python = raw"C:\Users\thoba\Documents\PlutoPythonTest\.venv\Scripts\python.exe"

ENV["JULIA_CONDAPKG_BACKEND"] = "Null"
ENV["JULIA_PYTHONCALL_EXE"] = pluto_python

In [ ]:
# Packages

using PythonCall
using Statistics
using FFTW
using LinearAlgebra
using Plots
using LibSerialPort

plotly()

adi = pyimport("adi")
subprocess = pyimport("subprocess")

In [ ]:
# ============================================================
# Radio settings
# ============================================================

sample_rate = 1_000_000           # samples
centre_frequency = 867_097_000    # Hz
rf_bandwidth = 800_000            # Hz
rx_buffer_size = 4096
rx_gain = 30.0                    # dB


# ============================================================
# Signal settings
# ============================================================

signal_frequency = 868_897_000    # Hz
signal_bandwidth = 200_000        # Hz

# ============================================================
# Array settings
# ============================================================

c = 3e8
wavelength = c / signal_frequency

antenna_spacing = 0.4             # metres

reference_antenna = "A1"

# HMC7992 switch path order
# path 1 [A2]
# path 2 [A3]
# path 3 [A4]
# path 4 [A5]

switched_antennas = ["A2", "A3", "A4", "A5"]

num_switched_antennas = length(switched_antennas)

# ============================================================
# STM32 settings
# ============================================================

stm_port = "COM8"
stm_baud = 115200

discard_buffers = 5

## Hardware Connections

In [ ]:
# Connect to the STM32 over the USB serial port

stm = LibSerialPort.open(stm_port, stm_baud)

println("STM32 connected on ", stm_port)

In [ ]:
# Find the Pluto Ethernet address

result = subprocess.run(
    ["iio_info", "-s"],
    capture_output = true,
    text = true
)

output = pyconvert(String, result.stdout)

pluto_ip = nothing

for line in split(output, '\n')

    if occursin("Pluto", line)

        for match in eachmatch(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", line)

            ip = match.match

            # Ignore the Pluto USB-network address
            if ip != "192.168.2.1"
                pluto_ip = ip
                break
            end
        end
    end
end

pluto_ip === nothing &&
    error("Pluto Ethernet connection not found.")

pluto_uri = "ip:" * pluto_ip

sdr = adi.ad9361(uri = pluto_uri)

println("Pluto connected at ", pluto_ip)

In [ ]:
function configure_rdf_receiver(sdr)

    # Use both receive channels
    pysetattr(sdr, "rx_enabled_channels", pylist([0, 1]))

    # RDF receiver settings
    pysetattr(sdr, "sample_rate", sample_rate)
    pysetattr(sdr, "rx_lo", centre_frequency)
    pysetattr(sdr, "rx_rf_bandwidth", rf_bandwidth)
    pysetattr(sdr, "rx_buffer_size", rx_buffer_size)

    # Manual gain on both channels
    pysetattr(sdr, "gain_control_mode_chan0", "manual")
    pysetattr(sdr, "gain_control_mode_chan1", "manual")

    pysetattr(sdr, "rx_hardwaregain_chan0", rx_gain)
    pysetattr(sdr, "rx_hardwaregain_chan1", rx_gain)

    println("Pluto configured for RDF measurements")

    return nothing
end

In [ ]:
configure_rdf_receiver(sdr)

## RF Switch Control

In [ ]:
# Select one of the switched antenna paths

function select_path(stm, path)
    
    # Check that the requested path exists
    if path < 1 ||  path > num_switched_antennas
        error("Invalid RF path: $path")
    end
    
    # Send the path number to the STM32
    command = string(path)
    write(stm, command)
    
    # Wait until the STM32 confirms the switched state
    response = strip(readline(stm))
    expected_response =  "READY $path"
    
    if response != expected_response
        error("Expected '$expected_response', received '$response'")
    end
    
    return nothing
end

## RDF Measurement Workflow

In [ ]:
# Measure RX2 relative to the fixed reference on RX1

function measure_relative_phase(sdr)
    
    # Capture both Pluto receive channels simultaneously
    rx_data = sdr.rx()
    
    rx1 = pyconvert(Vector{ComplexF64}, rx_data[0])
    rx2 = pyconvert(Vector{ComplexF64}, rx_data[1])
    
    # Remove DC offset
    rx1 = rx1 .- mean(rx1)
    rx2 = rx2 .- mean(rx2)
    
    # Relative complex phasor between RX2 and RX1
    cross_signal = sum(rx2 .* conj.(rx1))
    
    energy_rx1 = sum(abs2, rx1)
    energy_rx2 = sum(abs2, rx2)
    
    relative_phasor = cross_signal / sqrt(energy_rx1 * energy_rx2)
    
    phase_deg = rad2deg(angle(relative_phasor))
    correlation = abs(relative_phasor)
    gain_db = 10 * log10(energy_rx2 / energy_rx1)
    
    return (
        phase_deg = phase_deg,
        correlation = correlation,
        gain_db = gain_db  
    )
    
end

In [ ]:
# Measure all switched antenna paths relative to the fixed reference

function scan_switched_array(sdr, stm)
    
    phase_deg = zeros(Float64, num_switched_antennas)
    correlation = zeros(Float64, num_switched_antennas)
    gain_db = zeros(Float64, num_switched_antennas)
    
    for path in 1:num_switched_antennas
        
        # Selected the next antenna path
        select_path(stm, path)
        
        # Discard captures immediately after switching
        for _ in 1:discard_buffers
            sdr.rx()
        end
        
        # Measure RX2 relative to the fixed reference
        measurement = measure_relative_phase(sdr)
        
        phase_deg[path] = measurement.phase_deg
        correlation[path] = measurement.correlation
        gain_db[path] = measurement.gain_db
        
    end
    
    return (
        antennas = switched_antennas,
        phase_deg = phase_deg,
        correlation = correlation,
        gain_db = gain_db
    
    )

end

In [ ]:
# VCO peak detector


function detect_vco(
    rx1;
    fc,
    fs,
    search_low = 867.8e6,
    search_high = 868.8e6
)

    # --------------------------------------------------------
    # 1. Remove receiver DC offset
    # --------------------------------------------------------
    x = rx1 .- mean(rx1)

    N = length(x)

    # --------------------------------------------------------
    # 2. Apply Hann window
    # --------------------------------------------------------
    window = 0.5 .- 0.5 .* cos.(2π .* (0:N-1) ./ (N-1))

    # --------------------------------------------------------
    # 3. FFT
    # --------------------------------------------------------
    X = fftshift(fft(x .* window))

    # Power spectrum
    P = abs2.(X)

    # Frequency resolution
    df = fs / N

    # Absolute RF frequency axis
    freq = fc .+ (-N÷2:N÷2-1) .* df

    # --------------------------------------------------------
    # 4. Search only where the VCO should exist
    # --------------------------------------------------------
    search = (freq .>= search_low) .& (freq .<= search_high)

    search_indices = findall(search)

    # Strongest power bin inside search region
    idx = search_indices[argmax(P[search])]

    # --------------------------------------------------------
    # 5. Refine frequency using neighbouring FFT bins
    # --------------------------------------------------------
    delta = 0.0

    if 1 < idx < N

        y1 = log(P[idx-1] + 1e-20)
        y2 = log(P[idx]   + 1e-20)
        y3 = log(P[idx+1] + 1e-20)

        denominator = y1 - 2*y2 + y3

        if abs(denominator) > 1e-12
            delta = 0.5 * (y1 - y3) / denominator

            delta = clamp(delta, -0.5, 0.5)
        end
    end

    f_peak = freq[idx] + delta * df

    # --------------------------------------------------------
    # 6. Estimate local spectral floor
    # --------------------------------------------------------

    # Exclude bins immediately surrounding VCO
    exclude = max(first(search_indices), idx-10):min(last(search_indices), idx+10)

    floor_indices =
        setdiff(search_indices, collect(exclude))

    local_floor =
        median(P[floor_indices])

    peak_to_floor_db =
        10 * log10(
            (P[idx] + 1e-20) /
            (local_floor + 1e-20)
        )

    return (
        frequency = f_peak,
        baseband = f_peak - fc,
        peak_to_floor_db = peak_to_floor_db,
        spectrum = P,
        freq_axis = freq,
        fft_bin_width = df
    )
end

In [ ]:
# ============================================================
# Repeated relative-phase measurement
# ============================================================

path_to_measure = 4

# RF1 → A2
# RF2 → A3
# RF3 → A4
# RF4 → A5

current_path = select_path(stm, path_to_measure)

antenna_number = path_to_measure + 1

println(
    "Selected RF", path_to_measure,
    " → A", antenna_number
)

sleep(0.05)

num_measurements = 50
warmup_captures = 3

phases_deg = Float64[]
frequencies = Float64[]
ratios_db = Float64[]
peak_levels = Float64[]

fc = pyconvert(Float64, sdr.rx_lo)
fs = pyconvert(Float64, sdr.sample_rate)

# ------------------------------------------------------------
# Discard initial captures
# ------------------------------------------------------------

println("Discarding $warmup_captures warm-up captures...")

for _ in 1:warmup_captures
    sdr.rx()
end

println("Starting measurements...\n")

for k in 1:num_measurements

    # --------------------------------------------------------
    # Simultaneous RX1 / RX2 capture
    # --------------------------------------------------------

    rx_data = sdr.rx()

    rx1 = pyconvert(Vector{ComplexF64}, rx_data[0])
    rx2 = pyconvert(Vector{ComplexF64}, rx_data[1])

    # Remove DC offset
    rx1 = rx1 .- mean(rx1)
    rx2 = rx2 .- mean(rx2)

    N = length(rx1)

    # --------------------------------------------------------
    # Detect VCO from reference antenna A1
    # --------------------------------------------------------

    result = detect_vco(
        rx1;
        fc = fc,
        fs = fs,
        search_low = 867.8e6,
        search_high = 868.8e6
    )

    f_bb = result.baseband

    # --------------------------------------------------------
    # Extract same VCO tone from both channels
    # --------------------------------------------------------

    n = 0:N-1

    window =
        0.5 .- 0.5 .* cos.(2π .* n ./ (N-1))

    reference_tone =
        exp.(-1im .* 2π .* f_bb .* n ./ fs)

    Z1 = sum(rx1 .* window .* reference_tone)
    Z2 = sum(rx2 .* window .* reference_tone)

    # --------------------------------------------------------
    # Relative phase: A1 - A2
    # --------------------------------------------------------

    phase_deg =
        rad2deg(angle(Z1 * conj(Z2)))

    ratio_db =
        20 * log10(abs(Z2) / abs(Z1))

    push!(phases_deg, phase_deg)
    push!(frequencies, result.frequency)
    push!(ratios_db, ratio_db)
    push!(peak_levels, result.peak_to_floor_db)

    println(
        "Capture ", lpad(k, 2),
        " | f = ", round(result.frequency/1e6, digits=6), " MHz",
        " | phase = ", round(phase_deg, digits=2), "°",
        " | RX2/RX1 = ", round(ratio_db, digits=2), " dB",
        " | peak = ", round(result.peak_to_floor_db, digits=1), " dB"
    )
end

# ============================================================
# Circular phase statistics
# ============================================================

phases_rad = deg2rad.(phases_deg)

mean_vector =
    mean(exp.(1im .* phases_rad))

circular_mean_deg =
    rad2deg(angle(mean_vector))

R = abs(mean_vector)

circular_std_deg =
    rad2deg(sqrt(-2 * log(R)))

# ============================================================
# Frequency and magnitude statistics
# ============================================================

mean_frequency =
    mean(frequencies)

frequency_pp =
    maximum(frequencies) - minimum(frequencies)

mean_ratio_db =
    mean(ratios_db)

# ============================================================
# Summary
# ============================================================

println("\n==============================")
println("A1-A2 PHASE STABILITY SUMMARY")
println("==============================")

println(
    "Circular mean phase = ",
    round(circular_mean_deg, digits=2),
    "°"
)

println(
    "Circular phase std = ",
    round(circular_std_deg, digits=2),
    "°"
)

println(
    "Mean VCO frequency = ",
    round(mean_frequency/1e6, digits=6),
    " MHz"
)

println(
    "VCO peak-to-peak variation = ",
    round(frequency_pp/1e3, digits=2),
    " kHz"
)

println(
    "Mean RX2/RX1 = ",
    round(mean_ratio_db, digits=2),
    " dB"
)

In [ ]:
# ============================================================
# Five-angle measured array calibration
# ============================================================
cal_angles_deg = [0.0, 72.0, 144.0, 216.0, 288.0]

# Columns:
# A1-A2, A1-A3, A1-A4, A1-A5

cal_measured_deg = [
    -173.68    -9.56    -79.36   -172.69
     -22.72   100.47   -115.14    172.32
    -102.17   169.06    -73.11     35.99
     155.71   -45.97    -92.80    -72.41
    -133.64  -175.34     98.74     -9.68
]

cal_std_deg = [
    1.14   0.64   0.56   0.45
    1.16   0.74   0.36   0.96
    6.08   5.30   1.95   0.91
    0.94   2.59   1.92   2.57
    1.34   6.61   1.25   1.56
]

println("Calibration anchors loaded.")

In [ ]:
# ============================================================
# Ideal UCA phase vectors at calibration angles
# Convention: A1 - An
# ============================================================

cal_ideal_deg = [
     64.782973   169.604027   169.604027    64.782973
    -64.782973     0.000000   104.821053   104.821053
   -104.821053  -169.604027  -104.821053     0.000000
      0.000000  -104.821053  -169.604027  -104.821053
    104.821053   104.821053     0.000000   -64.782973
]

println("Ideal calibration vectors loaded.")

In [ ]:
# ============================================================
# Angle-dependent calibration corrections
# ============================================================

wrap_deg(x) = mod(x + 180, 360) - 180

cal_error_deg =
    wrap_deg.(cal_measured_deg .- cal_ideal_deg)

println("Calibration correction matrix:")
display(cal_error_deg)

In [ ]:
# ============================================================
# Circular interpolation of calibration corrections
# 1-degree lookup from 0° to 359°
# ============================================================

lookup_angles_deg = collect(0.0:1.0:359.0)

# Repeat the 0° point at 360° to close the circle
angles_ext = vcat(cal_angles_deg, 360.0)
errors_ext = vcat(
    cal_error_deg,
    permutedims(cal_error_deg[1, :])
)

cal_error_lookup_deg =
    zeros(length(lookup_angles_deg), size(cal_error_deg, 2))

for (i, θ) in enumerate(lookup_angles_deg)

    # Find the two neighbouring calibration anchors
    k = searchsortedlast(angles_ext, θ)
    k = min(k, length(cal_angles_deg))

    θ1 = angles_ext[k]
    θ2 = angles_ext[k + 1]

    t = (θ - θ1) / (θ2 - θ1)

    for path in 1:4

        e1 = errors_ext[k, path]
        e2 = errors_ext[k + 1, path]

        # Shortest circular change between the two corrections
        Δe = wrap_deg(e2 - e1)

        cal_error_lookup_deg[i, path] =
            wrap_deg(e1 + t * Δe)
    end
end

println("360° calibration correction lookup generated.")
println("Size = ", size(cal_error_lookup_deg))

In [ ]:
anchor_rows = Int.(cal_angles_deg) .+ 1

anchor_check =
    wrap_deg.(
        cal_error_lookup_deg[anchor_rows, :] .-
        cal_error_deg
    )

println("Interpolation error at measured anchors:")
display(anchor_check)

In [ ]:
@assert cal_angles_deg[1] == 0.0 "Calibration must start at exactly 0°"
@assert issorted(cal_angles_deg) "Calibration angles must be sorted"
@assert allunique(cal_angles_deg) "Calibration angles must be unique"

In [ ]:
println("359° correction = ", cal_error_lookup_deg[360, :])
println("0° correction   = ", cal_error_lookup_deg[1, :])

println(
    "Circular difference = ",
    wrap_deg.(cal_error_lookup_deg[360, :] .-
              cal_error_lookup_deg[1, :])
)

In [ ]:
# ============================================================
# Ideal 5-element UCA steering model
# Must match simulation notebook
# Convention: A1 - An
# ============================================================

uca_c = 3e8
uca_f = 868.1e6
uca_lambda = uca_c / uca_f
uca_radius = 0.09

uca_element_angles_deg = [0.0, 72.0, 144.0, 216.0, 288.0]

ideal_phase_lookup_deg = zeros(360, 4)

for theta_deg in 0:359

    theta = deg2rad(theta_deg)

    for n in 2:5

        alpha = deg2rad(uca_element_angles_deg[n])

        # Path difference for A1 - An
        path_difference =
            uca_radius * (cos(theta) - cos(theta - alpha))

        phase_rad =
            (2π / uca_lambda) * path_difference

        ideal_phase_lookup_deg[theta_deg + 1, n - 1] =
            wrap_deg(rad2deg(phase_rad))
    end
end

println("Ideal UCA lookup generated.")
println("Size = ", size(ideal_phase_lookup_deg))

In [ ]:
anchor_rows = [1, 73, 145, 217, 289]

println("Hardware-notebook ideal phases at anchors:")
display(ideal_phase_lookup_deg[anchor_rows, :])

println("Difference from stored simulation anchor values:")
display(
    wrap_deg.(
        ideal_phase_lookup_deg[anchor_rows, :] .-
        cal_ideal_deg
    )
)

In [ ]:
# ============================================================
# Final calibrated steering lookup
# ============================================================

calibrated_phase_lookup_deg =
    wrap_deg.(
        ideal_phase_lookup_deg .+
        cal_error_lookup_deg
    )

println("Final calibrated steering lookup generated.")
println("Size = ", size(calibrated_phase_lookup_deg))

In [ ]:
anchor_rows = Int.(cal_angles_deg) .+ 1

println("Calibrated lookup at measured anchors:")
display(calibrated_phase_lookup_deg[anchor_rows, :])

println("Difference from measured calibration data:")
display(
    wrap_deg.(
        calibrated_phase_lookup_deg[anchor_rows, :] .-
        cal_measured_deg
    )
)

In [ ]:
# ============================================================
# DoA estimator using calibrated steering lookup
# ============================================================

function estimate_doa(measured_phase_deg, calibrated_phase_lookup_deg)

    errors = zeros(360)

    for i in 1:360

        residual =
            wrap_deg.(
                measured_phase_deg .-
                calibrated_phase_lookup_deg[i, :]
            )

        errors[i] =
            sqrt(mean(residual .^ 2))
    end

    best_index = argmin(errors)

    estimated_angle_deg = best_index - 1
    minimum_error_deg = errors[best_index]

    return (
        angle_deg = estimated_angle_deg,
        error_deg = minimum_error_deg,
        errors = errors
    )
end

In [ ]:
for k in 1:length(cal_angles_deg)

    result = estimate_doa(
        cal_measured_deg[k, :],
        calibrated_phase_lookup_deg
    )

    println(
        "True = ", cal_angles_deg[k], "°",
        "   Estimated = ", result.angle_deg, "°",
        "   RMSE = ", round(result.error_deg, digits = 6), "°"
    )
end

In [ ]:
# ============================================================
# Analyse latest measurement run
# ============================================================

test_latest = [
   -157.52,
    -50.95,
    -94.37,
    164.76     # A1-A5
]

result = estimate_doa(
    test_latest,
    calibrated_phase_lookup_deg
)

println("Estimated angle = ", result.angle_deg, "°")
println("Minimum RMSE    = ",
        round(result.error_deg, digits = 2), "°")

best_row = result.angle_deg + 1

predicted_phase =
    calibrated_phase_lookup_deg[best_row, :]

residual =
    wrap_deg.(
        test_latest .- predicted_phase
    )

println("\nMeasured phase vector:")
display(test_latest)

println("Model phase vector at estimated angle:")
display(predicted_phase)

println("Per-path circular residuals:")
display(residual)

In [ ]:
# ============================================================
# Finer 10-point calibration grid
# 36° spacing
# ============================================================

cal10_angles_deg = collect(0.0:36.0:324.0)

cal10_measured_deg = fill(NaN, 10, 4)

# Copy the existing five official calibration anchors
# 0°, 72°, 144°, 216°, 288°
cal10_measured_deg[[1, 3, 5, 7, 9], :] .= cal_measured_deg

# Ideal UCA phases already available from the 360° model
cal10_ideal_deg =
    ideal_phase_lookup_deg[Int.(cal10_angles_deg) .+ 1, :]

println("10-point calibration grid prepared.")
println("Angles:")
display(cal10_angles_deg)

println("Measured table so far:")
display(cal10_measured_deg)

In [ ]:
# ============================================================
# Add 36° calibration anchor
# ============================================================

cal10_measured_deg[2, :] = [
     92.57,    # A1-A2
   -134.81,    # A1-A3
     19.32,    # A1-A4
   -170.99     # A1-A5
]

println("36° calibration anchor stored:")
display(cal10_measured_deg[2, :])

# ============================================================
# Add 108° calibration anchor
# ============================================================

cal10_measured_deg[4, :] = [
     -9.16,    # A1-A2
     34.48,    # A1-A3
    155.26,    # A1-A4
     45.36     # A1-A5
]

println("108° calibration anchor stored:")
display(cal10_measured_deg[4, :])

# ============================================================
# Add 180° calibration anchor
# ============================================================

cal10_measured_deg[6, :] = [
     49.32,
    -69.13,
    -22.89,
     66.61
]

println("180° calibration anchor stored:")
display(cal10_measured_deg[6, :])

# ============================================================
# Add 252° calibration anchor
# ============================================================

cal10_measured_deg[8, :] = [
   -172.46,    # A1-A2
     84.44,    # A1-A3
    -25.19,    # A1-A4
    -50.80     # A1-A5
]

println("252° calibration anchor stored:")
display(cal10_measured_deg[8, :])

# ============================================================
# Add 324° calibration anchor
# ============================================================

cal10_measured_deg[10, :] = [
   -126.36,    # A1-A2
    -21.55,    # A1-A3
   -107.65,    # A1-A4
    116.37     # A1-A5
]

println("324° calibration anchor stored:")
display(cal10_measured_deg[10, :])

In [ ]:
# ============================================================
# Build 10-anchor calibration residuals
# ============================================================

@assert !any(isnan, cal10_measured_deg) "Calibration table still contains NaN values"

cal10_error_deg =
    wrap_deg.(cal10_measured_deg .- cal10_ideal_deg)

println("10-point calibration residuals:")
display(cal10_error_deg)

In [ ]:
# ============================================================
# Interpolate 10-anchor calibration residuals over 0–359°
# ============================================================

function circular_error_lookup(cal_angles_deg, cal_error_deg)

    @assert cal_angles_deg[1] == 0.0
    @assert issorted(cal_angles_deg)
    @assert allunique(cal_angles_deg)

    angles_ext = vcat(cal_angles_deg, 360.0)

    errors_ext = vcat(
        cal_error_deg,
        permutedims(cal_error_deg[1, :])
    )

    lookup = zeros(360, size(cal_error_deg, 2))

    for θ in 0:359

        i = θ + 1

        k = searchsortedlast(angles_ext, θ)
        k = min(k, length(cal_angles_deg))

        θ1 = angles_ext[k]
        θ2 = angles_ext[k + 1]

        t = (θ - θ1) / (θ2 - θ1)

        for path in 1:size(cal_error_deg, 2)

            e1 = errors_ext[k, path]
            e2 = errors_ext[k + 1, path]

            Δe = wrap_deg(e2 - e1)

            lookup[i, path] =
                wrap_deg(e1 + t * Δe)
        end
    end

    return lookup
end

cal10_error_lookup_deg =
    circular_error_lookup(
        cal10_angles_deg,
        cal10_error_deg
    )

size(cal10_error_lookup_deg)

In [ ]:
# ============================================================
# Build 10-point calibrated steering lookup
# ============================================================

calibrated10_phase_lookup_deg =
    wrap_deg.(
        ideal_phase_lookup_deg .+
        cal10_error_lookup_deg
    )

println("Calibrated steering table size:")
display(size(calibrated10_phase_lookup_deg))

In [ ]:
# ============================================================
# Verify all 10 calibration anchors
# ============================================================

anchor_indices = Int.(cal10_angles_deg) .+ 1

model_at_anchors =
    calibrated10_phase_lookup_deg[anchor_indices, :]

anchor_errors =
    wrap_deg.(model_at_anchors .- cal10_measured_deg)

println("Anchor reproduction errors (degrees):")
display(anchor_errors)

println(
    "\nMaximum absolute anchor error = ",
    maximum(abs.(anchor_errors)),
    "°"
)

In [ ]:
# ============================================================
# Test an off-anchor measurement
# ============================================================

measured_test_deg = [
    49.99,
    -162.55,
    -39.19,
    -174.44
]

result =
    estimate_doa(
        measured_test_deg,
        calibrated10_phase_lookup_deg
    )

println("Estimated DoA = ", result.angle_deg, "°")
println("Minimum phase RMSE = ", round(result.error_deg, digits=2), "°")

In [ ]:
model_86_deg =
    calibrated10_phase_lookup_deg[86 + 1, :]

residual_86_deg =
    wrap_deg.(measured_test_deg .- model_86_deg)

println("Model at 86°:")
display(model_86_deg)

println("Residuals at estimated angle:")
display(residual_86_deg)

## Optional Utility: VCO Source Characterisation

The VCO carrier frequency is not known exactly because it depends on the
tuning voltage. The Pluto is therefore swept across a wide frequency range
and the received spectrum is examined to locate the strongest RF carrier.

In [ ]:
# Frequency range to search for the VCO carrier

sweep_start = 650_000_000       # 650 MHz
sweep_stop  = 1_100_000_000     # 1100 MHz

sweep_sample_rate = 6_000_000   # 5 MS/s
sweep_bandwidth = 5_000_000     # 4 MHz
sweep_step = 5_000_000          # Move 4 MHz at a time

sweep_buffer_size = 4096

# Centre frequencies that the Pluto will visit during the sweep

sweep_frequencies = sweep_start : sweep_step : sweep_stop

In [ ]:
# Configure Pluto for the VCO frequency sweep

sweep_gain = 10.0     

pysetattr(sdr, "rx_enabled_channels", pylist([0]))
pysetattr(sdr, "sample_rate", sweep_sample_rate)
pysetattr(sdr, "rx_rf_bandwidth", sweep_bandwidth)
pysetattr(sdr, "rx_buffer_size", sweep_buffer_size)

pysetattr(sdr, "gain_control_mode_chan0", "manual")
pysetattr(sdr, "rx_hardwaregain_chan0", sweep_gain)

println("Sweep sample rate: ", sweep_sample_rate / 1e6, " MS/s")
println("Sweep bandwidth:   ", sweep_bandwidth / 1e6, " MHz")
println("Sweep RX gain:     ", sweep_gain, " dB")

In [ ]:
# Storage for the complete measured spectrum

all_frequencies = Float64[]
all_magnitudes = Float64[]


for centre_freq in sweep_frequencies
    
    # Tune Pluto to the next centre frequency
    pysetattr(sdr, "rx_lo", centre_freq)
    
    # Discard one capture after changing frequency
    # so that the next capture belongs to the new tuning position
    sdr.rx()
    
    # Capture IQ samples
    rx_data = sdr.rx()
    rx = pyconvert(Vector{ComplexF64}, rx_data)
    
    # Remove DC offset
    rx = rx .- mean(rx)
    
    # Calculate the spectrum
    N = length(rx)
    
    frequency_bins = -N÷2 : N÷2-1
    baseband_frequency = frequency_bins .* (sweep_sample_rate / N)
    
    spectrum = abs.(fftshift(fft(rx)))
    
    # Convert baseband frequency to actual RF frequency
    rf_frequency = centre_freq .+ baseband_frequency

    # Keep only the central 4 MHz of this capture
    keep = abs.(baseband_frequency) .<= sweep_bandwidth / 2

    append!(all_frequencies, rf_frequency[keep])
    append!(all_magnitudes, spectrum[keep])
end

println("Sweep complete")

In [ ]:
peak_index = argmax(all_magnitudes)

peak_frequency = all_frequencies[peak_index]
peak_magnitude = all_magnitudes[peak_index]

println("Strongest detected frequency: ", round(peak_frequency / 1e6, digits = 3), " MHz")

In [ ]:
# Plot the FFT magnitude across the complete sweep

plot(
    all_frequencies ./ 1e6,
    all_magnitudes,

    xlabel = "Frequency (MHz)",
    ylabel = "FFT Magnitude",
    title = "VCO Frequency Spectrum",

    legend = false,
    linewidth = 1
)

In [ ]:
# Restore the Pluto to the normal RDF receiver configuration
configure_rdf_receiver(sdr)